# V1 — Per-Prompt Activation Vector Extraction

Saves individual prompt activation vectors (not aggregated sums) for probe training.
Outputs one HDF5 per cell: `{PREFIX}V1_vectors.h5` with datasets
`{concept_type}/{concept}/layer_{L}` of shape `(n_prompts, mlp_dim)` in float16.

**GPU required.** Run once per cell (P_QW first, then R_QW if time).

In [ ]:
# Cell 1 – Dependencies
import subprocess, sys, os

pkgs = ["h5py", "transformers", "torch", "numpy", "pandas", "tqdm",
        "pyarrow", "accelerate"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

print("Dependencies ready")

In [ ]:
# Cell 2 – Configuration
LANG = "P"
MODEL = "QW"
PREFIX = f"{LANG}_{MODEL}_"
MODEL_CONFIGS = {
    "QW": {"id": "Qwen/Qwen2.5-Coder-7B",                "n_layers": 28, "mlp_dim": 3584},
    "DS": {"id": "deepseek-ai/deepseek-coder-6.7b-base",  "n_layers": 32, "mlp_dim": 4096},
}
MODEL_ID = MODEL_CONFIGS[MODEL]["id"]
N_LAYERS = MODEL_CONFIGS[MODEL]["n_layers"]
MLP_DIM = MODEL_CONFIGS[MODEL]["mlp_dim"]

OBJECT_PROMPTS = f"{PREFIX}1A_object_prompts.parquet"
CHECKER_PROMPTS = f"{PREFIX}1B_checker_prompts.parquet"
OUTPUT_FILE = f"{PREFIX}V1_vectors.h5"

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/New-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/New-Atlas"

LOCAL_SRC = "/Users/piotrwilam/Code/New-Atlas/src"
COLAB_SRC = "/content/drive/MyDrive/CODE/New-Atlas/src"
SRC_PATH = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

os.makedirs(DATA_DIR, exist_ok=True)

print(f"Cell: {LANG}_{MODEL} | Model: {MODEL_ID}")
print(f"Layers: {N_LAYERS} | MLP dim: {MLP_DIM}")
print(f"Output: {OUTPUT_FILE}")

In [ ]:
# Cell 3 – Imports & model loading
import numpy as np
import pandas as pd
import torch
import h5py
import re
import logging
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("V1")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
).eval()

print(f"Model loaded | Device: {DEVICE}")

In [ ]:
# Cell 4 – Register hooks
from module2.extraction import ActivationExtractor

mlp_pattern = None
for name, _ in model.named_modules():
    m = re.match(r'^(.*?)(\d+)(\.mlp)$', name)
    if m:
        if mlp_pattern is None:
            mlp_pattern = m.group(1) + "{layer_id}" + m.group(3)
            break

print(f"Pattern: {mlp_pattern}")

extractor = ActivationExtractor(
    model=model, tokenizer=tokenizer, device=DEVICE,
    n_layers=N_LAYERS, use_hook_recorder=False,
)
extractor.set_hook_pattern(mlp_pattern)
extractor.register_hooks()

# Sanity check
test_code = 'x = 42' if LANG == 'P' else 'fn main() { let x = 42; }'
acts = extractor.extract(test_code)
print(f"Sanity check: {len(acts)} layers, dim={acts[0].shape}")

In [ ]:
# Cell 5 – Load prompts and identify testable concepts

# Object prompts
df_obj = pd.read_parquet(os.path.join(DATA_DIR, OBJECT_PROMPTS))
if LANG == "P":
    # Python: group by (ast_node, builtin_obj), universal = marginalise over builtins
    # For probes we need per-concept prompts, not per-pair
    # Use all prompts where this concept appears (across all pairings)
    obj_col = "ast_node"
    pair_col = "builtin_obj"
else:
    obj_col = "construct"
    pair_col = "object"

print(f"Object prompts: {len(df_obj)} rows")

# Checker prompts
df_chk = pd.read_parquet(os.path.join(DATA_DIR, CHECKER_PROMPTS))
print(f"Checker prompts: {len(df_chk)} rows")

# Identify testable concepts: those present in BOTH object and checker
if LANG == "P":
    obj_concepts = set(df_obj[obj_col].unique())
    chk_concepts = set()
    for obj_key in df_chk["object"].unique():
        parts = obj_key.split("__", 1)
        if parts[0] == "ast":
            chk_concepts.add(parts[1])
        elif parts[0] == "builtin":
            chk_concepts.add(parts[1])
    testable = sorted(obj_concepts & chk_concepts)
    # Build checker key lookup
    chk_key_map = {}  # concept_name -> checker object key
    for obj_key in df_chk["object"].unique():
        parts = obj_key.split("__", 1)
        chk_key_map[parts[1]] = obj_key
else:
    # Rust: construct display names in object prompts, rust__Name in checker
    obj_concepts = set(df_obj[obj_col].unique())
    chk_concepts_raw = set(df_chk["object"].unique())
    chk_concepts = set()
    chk_key_map = {}
    for obj_key in chk_concepts_raw:
        parts = obj_key.split("__", 1)
        if len(parts) == 2:
            chk_concepts.add(parts[1])
            chk_key_map[parts[1]] = obj_key
    testable = sorted(obj_concepts & chk_concepts)

print(f"Testable concepts: {len(testable)}")
print(f"Sample: {testable[:10]}")

In [ ]:
# Cell 6 – Extract and save per-prompt vectors
import time

out_path = os.path.join(DATA_DIR, OUTPUT_FILE)
start = time.time()

with h5py.File(out_path, "w") as f:
    # Metadata
    md = f.create_group("metadata")
    md.attrs["model_id"] = MODEL_ID
    md.attrs["lang"] = LANG
    md.attrs["model"] = MODEL
    md.attrs["n_layers"] = N_LAYERS
    md.attrs["mlp_dim"] = MLP_DIM

    for concept_idx, concept in enumerate(tqdm(testable, desc="Concepts")):

        # ── Object prompts for this concept ──
        obj_prompts = df_obj[df_obj[obj_col] == concept]["prompt_text"].tolist()
        if len(obj_prompts) == 0:
            logger.warning(f"No object prompts for {concept}")
            continue

        # Extract one prompt at a time (safe for memory)
        obj_vectors = {lid: [] for lid in range(N_LAYERS)}
        for prompt in obj_prompts:
            acts = extractor.extract(prompt, token_pos=-1)
            for lid, vec in acts.items():
                obj_vectors[lid].append(vec.half().numpy())

        for lid in range(N_LAYERS):
            if obj_vectors[lid]:
                arr = np.stack(obj_vectors[lid])  # (n_prompts, mlp_dim) float16
                f.create_dataset(
                    f"object/{concept}/layer_{lid}",
                    data=arr, compression="gzip", compression_opts=1
                )
        del obj_vectors

        # ── Checker prompts for this concept ──
        chk_key = chk_key_map.get(concept)
        if chk_key is None:
            continue
        chk_prompts = df_chk[df_chk["object"] == chk_key]["prompt_text"].tolist()
        if len(chk_prompts) == 0:
            continue

        chk_vectors = {lid: [] for lid in range(N_LAYERS)}
        for prompt in chk_prompts:
            acts = extractor.extract(prompt, token_pos=-1)
            for lid, vec in acts.items():
                chk_vectors[lid].append(vec.half().numpy())

        for lid in range(N_LAYERS):
            if chk_vectors[lid]:
                arr = np.stack(chk_vectors[lid])
                f.create_dataset(
                    f"checker/{concept}/layer_{lid}",
                    data=arr, compression="gzip", compression_opts=1
                )
        del chk_vectors

        if (concept_idx + 1) % 10 == 0:
            elapsed = time.time() - start
            print(f"  [{concept_idx+1}/{len(testable)}] {elapsed/60:.1f} min")

elapsed = time.time() - start
file_size = os.path.getsize(out_path) / 1e6
print(f"\nDone: {out_path}")
print(f"Size: {file_size:.0f} MB | Time: {elapsed/60:.1f} min")

In [ ]:
# Cell 7 – Verify output
with h5py.File(out_path, "r") as f:
    n_obj = len(f["object"]) if "object" in f else 0
    n_chk = len(f["checker"]) if "checker" in f else 0
    print(f"Object concepts: {n_obj}")
    print(f"Checker concepts: {n_chk}")

    # Sample one concept
    if n_obj > 0:
        sample_concept = list(f["object"].keys())[0]
        sample_data = f[f"object/{sample_concept}/layer_0"]
        print(f"\nSample: object/{sample_concept}/layer_0")
        print(f"  Shape: {sample_data.shape} | Dtype: {sample_data.dtype}")
        vals = sample_data[:]
        print(f"  Min: {vals.min():.4f} | Max: {vals.max():.4f} | Mean: {vals.mean():.4f}")

In [ ]:
# Cell 8 – Cleanup
extractor.remove_hooks()
print("Hooks removed.")
print(f"\nV1 complete: {OUTPUT_FILE}")

try:
    from google.colab import runtime
    runtime.unassign()
except (ImportError, AttributeError):
    print("Not running on Colab -- skipping unassign.")